## Signal or Noise? Teaching a Computer to Find Pulsars

This activity uses data from the [High Time Resolution Universe survey](https://www.mpifr-bonn.mpg.de/research/fundamental/htru), a radio telescope search for pulsars. It's the companion to the Invariant Mass of the Muon activity, and it's building toward classifying neutrino events in the NOvA detector.  

To get started,
- You won't hurt anything by experimenting. If you break it, close the tab and open the activity again to start over.
- Did you do the muon notebook? You'll want it. This one picks up exactly where that one's window-drawing left off.  It could be helpful to have it open in a nearby tab for reference as you work.

**Where we're headed:** In the muon notebook, you drew a window around a peak by hand and kept what was inside. That's called making a *cut*, and physicists have done it that way for a hundred years. Here, you'll do it by hand one more time, and then you'll teach a computer to do it for you — and to do it in eight dimensions at once, which would be a much greater challenge for a human to accomplish.

When you're ready, run each code cell until you get down to **Part One**.

In [ ]:
# imports some software packages we'll use
import pandas as pd #For data organization
import numpy as np #For Math
import matplotlib as mpl #For plotting
import matplotlib.pyplot as plt #Also for plotting

# These are new! This is scikit-learn, the machine learning toolkit.
from sklearn.tree import DecisionTreeClassifier, plot_tree #The model we'll train
from sklearn.model_selection import train_test_split #For splitting our data honestly
from sklearn.metrics import confusion_matrix #For grading our model

print("Packages Imported!")

In [ ]:
# Here we read in the pulsar data set, hosted on Google Drive
data = pd.read_csv('pulsarData.csv')
data.head()

In [ ]:
# A call to "describe" gives an overview of the data
data.describe()

## Part One
Get acquainted with this data set. Each row is a **candidate**: a blip the radio telescope saw that *might* be a pulsar.

A pulsar is a neutron star that spins fast and sweeps a beam of radio waves past Earth, like a lighthouse. They're scientifically precious — we use them as clocks, as probes of spacetime, and as tests of general relativity. They're also rare, and radio telescopes see a *lot* of junk: cell phones, microwaves, lightning, satellites. Astronomers call that junk **RFI** (radio frequency interference).

So somebody has to look at every candidate and decide: real pulsar, or junk? There are too many to do by hand. That's the problem we're solving.

**Update and Run** the cell below.  Use it's results to answer the following questions:

- How many candidates does this data set contain?
- The `target` column is the answer key: 1 means a real pulsar, 0 means junk. How many of each are there?



In [ ]:
# How many total events & how many of each kind of candidate do we have?
totalNumEvents = len(data['target']) # len returns how many lines are in a column
numPulsars = np.sum(data['t_____']) # remember, a value of 1 is a pulsar
numJunk = ***some math here using above variables to determine number of junk events***

print("Number of Events", totalNumEvents)
print("Number of Pulsars", numPul____)
print("Number of /'Junk/' events", numJu_____)

## Part Two: A Trap, Right Out of the Gate
Look hard at that last number. About **91%** of these candidates are junk. Only about 9% are real pulsars.

Now here's the trap.  We could literally just add up the number of events and say that they are *ALL JUNK* without any additional logic

That's it. It says *nothing is ever a pulsar*, every single time, without even looking at the data. **How often is our lazy detector right?**

About 91% of the time. It is 91% accurate. It has also never found a pulsar in its life and never will.  

- Would you call that detector good? Why not?
- What is a 91% as a grade a student could earn in a class you teach?
- What's wrong with "accuracy" as a way of grading?

Hang onto that 91% number. Every time you see a score in this notebook, ask yourself: *is that actually better than the lazy detector?* This is not a trick question about coding — it's a real problem that real astronomers and real particle physicists have to think about constantly, because **the interesting thing is almost always the rare thing.** NOvA has this exact problem. So does every search for a new particle ever conducted.

## Part Three: What Are These Columns, Physically?
Eight columns, and they're not eight random measurements. They're two groups of four, and each group is the *same four statistics* (mean, standard deviation, excess kurtosis, and skewness) computed on a different thing. Let's take them one at a time, because you can reason your way to the answer here before you ever run a model.

**The integrated pulse profile** (`mean_prof`, `std_prof`, `kurt_prof`, `skew_prof`)  
A pulsar's beam sweeps past us over and over. If you fold all those rotations on top of each other, a real pulsar gives you a **sharp, narrow spike** sitting on a flat baseline — its fingerprint. RFI gives you a broad, sloppy mess.  
- *Kurtosis* measures how spiky a distribution is. *Skewness* measures how lopsided it is.
- **Predict:** should a real pulsar have HIGH or LOW kurtosis compared to junk? Talk it out with a partner before you look.

**The DM-SNR curve** (`mean_dmsnr`, `std_dmsnr`, `kurt_dmsnr`, `skew_dmsnr`)  
This one is better physics. Radio waves from a pulsar cross thousands of light years of interstellar space, which is full of free electrons. The waves are electromagnetic, so they interact with those electrons and get slowed down — and here's the key: **the amount of slowing depends on frequency.** Low frequencies arrive *later* than high frequencies. This is called **dispersion**, and the *dispersion measure* (DM) tells you roughly how much electron-filled space the signal crossed. It's a stand-in for distance.

To find a pulsar you try un-dispersing the signal at many trial DM values and see which one makes the pulse snap into focus. Plot signal-to-noise vs. DM and you get the DM-SNR curve.
- A real pulsar is far away, so its curve peaks at some **DM greater than zero.**
- Your microwave oven is in the building. It crossed no interstellar space at all. **Its DM is zero.**

Read that last pair again, because it's the whole ballgame. The junk isn't junk because it's rare. **The junk is junk because a physical process that happens to the signal didn't happen to it.** Keep that in your pocket for NOvA — that's *exactly* why a neutral current event looks different from a muon neutrino event. Something the signal does, the background doesn't.

## Part Four: Look Before You Model
Never, ever fit a model to data you haven't looked at. Let's plot one feature for pulsars and junk separately and see if they actually separate.
- The code below plots `kurt_prof`. Run it.
- Was your prediction from Part Three right?
- Now edit the code to try some other features. Which ones separate the two groups well? Which ones are useless?

In [ ]:
featureToPlot = 'kurt_prof'   # <-- change this to explore other columns!

pulsars = data[data['target'] == 1]
junk = data[data['target'] == 0]

plt.hist(junk[featureToPlot], bins=50, alpha=0.6, label='junk (RFI)', density=True)
plt.hist(pulsars[featureToPlot], bins=50, alpha=0.6, label='real pulsars', density=True)
plt.xlabel(featureToPlot)
plt.ylabel("fraction of candidates")
plt.title("Change this title to something better")
plt.legend()
plt.grid(False);

# Why density=True? Try taking it out and see what happens.
# There are 10x more junk candidates than pulsars, so on a raw count plot
# the pulsars are a flea next to an elephant. This puts them on equal footing.

## Part Five: Draw the Cut Yourself
Remember the muon notebook? You drew a window around the mass peak, kept what was inside, and counted. Do it again — same move, different science.

Looking at your histogram, **pick a cut value on `kurt_prof`.** Everything above it, you're calling a pulsar. Everything below, junk. Then we'll grade you with two numbers:

- **Efficiency** = of all the real pulsars out there, what fraction did you catch? *(Did you miss any?)*
- **Purity** = of everything you called a pulsar, what fraction really was one? *(Did you cry wolf?)*

These are the same two numbers that fought each other in the muon notebook, and they're the two numbers NOvA quotes. Fill in the blank and run it.
- Try to get efficiency above 0.90. What happened to purity?
- Now try to get purity above 0.95. What happened to efficiency?
- **Can you get both above 0.95?** Really try. This matters more than getting it right.

In [ ]:
myCut = **ValueHere**     # <-- your cut value on kurt_prof. Everything above this, you're calling a pulsar.

iCalledItAPulsar = data['kurt_prof'] > myCut
itReallyIsAPulsar = data['target'] == 1

truePositives  = np.sum(iCalledItAPulsar & itReallyIsAPulsar)    # caught it!
falsePositives = np.sum(iCalledItAPulsar & ~itReallyIsAPulsar)   # cried wolf...the ~ sign acts as a "not" operator
falseNegatives = np.sum(~iCalledItAPulsar & itReallyIsAPulsar)   # missed a real one

efficiency = truePositives / (truePositives + falseNegatives)
purity = truePositives / (truePositives + falsePositives)

print("Pulsars I caught:      ", truePositives)
print("Junk I mistook:        ", falsePositives)
print("Pulsars I missed:      ", falseNegatives)
print()
print("Efficiency:", round(efficiency, 3))
print("Purity:    ", round(purity, 3))

## Part Six: Let the Computer Draw It
Now the reveal. We're going to train a **decision tree** — and we're going to hobble it on purpose. `max_depth=1` means it's allowed to ask exactly **one question** about one variable. It gets to pick which variable and where to cut. That's all.

In other words: we're asking the computer to do *precisely* what you just did by hand.
- Run it. Which variable did it choose? Where did it put the cut?
- How close is that to the number you picked?
- Did it beat you?

In [ ]:
features = ['mean_prof', 'std_prof', 'kurt_prof', 'skew_prof',
            'mean_dmsnr', 'std_dmsnr', 'kurt_dmsnr', 'skew_dmsnr']

X = data[features]     # the measurements. Capital X by convention.
y = data['target']     # the answer key. Lowercase y by convention.

stump = DecisionTreeClassifier(max_depth=1, random_state=42)   # only ONE question allowed
stump.fit(X, y)                                               # "fit" means "learn"

whichFeature = features[stump.tree_.feature[0]]
whereItCut = stump.tree_.threshold[0]

print("The computer chose to cut on:", whichFeature)
print("It put the cut at:", round(whereItCut, 3))
print()
print("Your cut was at:", myCut)

So a machine learning model is not magic, and it is not a brain. **A decision tree is a machine that finds cuts.** You already knew how to do this. It just does it faster, and it never gets tired, and — as we're about to see — it can ask follow-up questions.

That's worth sitting with for a second. Nothing spooky happened. No intelligence was involved. It tried a bunch of cut values, scored each one, and kept the best. You could do it with a ruler and a lot of patience.

## Part Seven: Training Data, and Why You Can't Grade Your Own Homework
Here's a rule that will feel obvious once you hear it, and that people violate constantly:

> **You cannot use the same data to tune your cut and to report how well your cut works.**

If you tune a window on some events and then count those same events, you haven't measured your method. You've measured your own tuning.

So we split the data in two:
- The **training set** is what the model gets to learn from. It sees the answers.
- The **testing set** we lock in a drawer. The model never sees it during training. That's our honest exam.

Two details in the code below are worth more than a passing glance:
- `stratify=y` forces both halves to keep the same 9% pulsar fraction. Without it, a random split might hand one half a different pulsar rate than the real sky has — and then you'd be testing on a universe that doesn't exist.
- `random_state=42` makes the split reproducible, so everyone in the room gets the same answer. **Try changing it to 1, then 2, then 3.** The scores wobble a little. That wobble is real — it's statistical uncertainty on your measurement, and it's what error bars are for.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,        # lock away 30% for the exam
    random_state=42,      # try changing me to 1, 2, 3...
    stratify=y            # keep the 9% pulsar fraction in both halves
)

# We print outputs to verify our training splits and fractions of candidates
print("Training candidates:", len(X_train))
print("Testing candidates: ", len(X_test))
print()
print("Pulsar fraction in training set:", round(y_train.mean(), 4))
print("Pulsar fraction in testing set: ", round(y_test.mean(), 4))

## Part Eight: Let's Break It On Purpose
Now let's take the leash off. `DecisionTreeClassifier()` with no `max_depth` can ask as many questions as it wants — it can keep subdividing until every single training candidate is sorted perfectly.

That sounds great. **Predict what will happen** to the two scores below before you run it. Write your prediction down.

In [ ]:
greedyTree = DecisionTreeClassifier(random_state=42)   # no limit! ask as many questions as you like!
greedyTree.fit(X_train, y_train)

print("Accuracy on the TRAINING data:", round(greedyTree.score(X_train, y_train), 4))
print("Accuracy on the TESTING data: ", round(greedyTree.score(X_test, y_test), 4))
print()
print("How many questions deep did it go?", greedyTree.get_depth())

**Following running the cell above, make some comments on your predictions and the outputs, then read on...**

The tree didn't learn what a pulsar *is*. It memorized the specific candidates you showed it — down to the noise in each one. That gap between the training score and the testing score has a name: **overfitting**. It's the single most common way to fool yourself in machine learning, and now you've seen it happen with your own eyes instead of being lectured about it.

Now let's put the leash back on and only allow 3 questions.
- **Predict again:** the training accuracy will go *down*. Will the testing accuracy go up or down?

In [ ]:
tree = DecisionTreeClassifier(max_depth=3, random_state=42)   # only 3 questions deep
tree.fit(X_train, y_train)

print("Accuracy on the TRAINING data:", round(tree.score(X_train, y_train), 4))
print("Accuracy on the TESTING data: ", round(tree.score(X_test, y_test), 4))
print()
print("...and remember, the lazy detector scores", round(1 - y.mean(), 4))

Read those numbers carefully. **The simpler model fits the training data worse and performs better.** That's not a typo and it's not a paradox. A model that's too flexible chases noise. Restraint is a feature.

And check it against the lazy detector, like we promised in Part Two. Are we actually beating 0.908 by enough to be proud of?

## Part Nine: Read the Tree Out Loud
This is why we chose a decision tree instead of something fancier. **You can just look at it.**

Run the cell, then try to read the top node out loud as a sentence in plain English. It's a cut on a physical measurement — the same kind of statement you'd write in a lab notebook.
- Does the tree's first question match what you'd have picked from the physics in Part Three?
- Follow one path from the top all the way down to a leaf. Can you narrate the whole decision?

In [ ]:
plt.figure(figsize=(18, 9))
plot_tree(tree,
          feature_names=features,
          class_names=['junk', 'PULSAR'],
          filled=True, rounded=True, fontsize=9)
plt.show()

# 'value' in each box shows [how many junk, how many pulsars] landed there.
# 'gini' measures how mixed-up a box is. 0.0 means perfectly sorted.

## Part Ten: The Confusion Matrix
Accuracy hides the story. Let's see the four numbers that actually matter — the same four you computed by hand in Part Five, now on the locked-away testing data.

In [ ]:
predictions = tree.predict(X_test)
cm = confusion_matrix(y_test, predictions)
trueNeg, falsePos, falseNeg, truePos = cm.ravel()

print("Correctly called junk:        ", trueNeg)
print("Junk we mistook for a pulsar: ", falsePos, "  <-- wasted telescope time")
print("Pulsars we MISSED:            ", falseNeg, "  <-- a discovery, thrown in the trash")
print("Pulsars we caught:            ", truePos)
print()
print("Efficiency:", round(truePos / (truePos + falseNeg), 3))
print("Purity:    ", round(truePos / (truePos + falsePos), 3))

Look at those two mistakes. They are **not** the same mistake.
- A false positive costs you a few minutes of follow-up telescope time.
- A false negative is a real pulsar you threw away and will never look at again.

Which one should we work harder to avoid? **That's not a math question.** No equation answers it. It depends on what you're trying to do, and a human being has to decide. Machine learning does not get you out of making that call — it just makes the call explicit.

## Part Eleven: Picking Your Working Point
Here's the thing nobody tells you about classifiers: **the model doesn't actually output "pulsar" or "junk."** It outputs a *number* between 0 and 1 — a confidence score. Somebody has to decide how confident is confident enough. That's a **threshold**, and it's a choice, not a result.

By default `predict()` uses 0.5. There's nothing sacred about 0.5.
- Run the cell. Watch efficiency and purity trade off against each other.
- This is the *exact* same tug-of-war as widening your mass window in the muon notebook. Same physics, same tradeoff, fancier tool.
- **Which row would you pick, and why?** Defend it. Then find somebody who picked a different row and argue about it — that argument is the actual job.

In [ ]:
probabilities = tree.predict_proba(X_test)[:, 1]   # the model's confidence that each candidate is a pulsar

print("threshold | efficiency | purity")
print("----------|------------|-------")
for threshold in [0.1, 0.3, 0.5, 0.7, 0.9]:
    calledPulsar = probabilities > threshold
    tp = np.sum(calledPulsar & (y_test == 1))
    fp = np.sum(calledPulsar & (y_test == 0))
    fn = np.sum(~calledPulsar & (y_test == 1))
    eff = tp / (tp + fn) if (tp + fn) > 0 else 0
    pur = tp / (tp + fp) if (tp + fp) > 0 else float('nan')
    print(f"   {threshold}    |   {eff:.3f}    | {pur:.3f}")

# Loosen the threshold: catch more pulsars, but drag in more junk.
# Tighten it: cleaner sample, but discoveries slip through your fingers.
# There is no setting that wins on both. There never is.

## Part Twelve: Which Measurement Did the Work?
Last question, and it's a physics question, not a coding one. The tree had eight measurements to choose from. **Which ones did it actually lean on?**

In [ ]:
importance = pd.DataFrame({
    'feature': features,
    'importance': tree.feature_importances_
}).sort_values('importance', ascending=False)

print(importance.to_string(index=False))

- Which features did the tree ignore completely? Why might that be?
- Does the ranking match the physics you reasoned about in Part Three?
- If a feature has zero importance, does that mean it carries no information about pulsars? *(Careful — it might just be saying the same thing as a feature the tree already used. Redundant is not the same as useless.)*

This is the payoff of using a model you can see inside. You didn't just get a prediction — you got a claim about *which measurement matters*, and that's something you can go argue about with an astronomer.

## Part Thirteen: On to NOvA
Take one more look at your code. Now imagine changing three things:

| this notebook | the NOvA notebook |
|---|---|
| a radio candidate | a neutrino interaction in the detector |
| `kurt_prof`, `mean_dmsnr`, ... | track length, energy deposited, number of hits |
| pulsar vs. RFI | muon neutrino charged current vs. neutral current |
| ~9% are pulsars | the interesting events are rare too |

**That's it. That's the whole change.** The `fit`, the `predict`, the split, the tradeoff — every line survives. That's not a coincidence or a cute analogy. It's the actual reason this method took over experimental physics: the problem shape is the same everywhere.

---  
## Saving Your Work  
This is running on a Google server on a distant planet and deletes what you've done when you close this tab. To save your work for later use or analysis you have a few options:  
- File > "Save a copy in Drive" will save it to you Google Drive in a folder called "Collaboratory". You can run it later from there.  
- File > "Download .ipynb" to save to your computer (and run with Jupyter software later)  
- File > Print to ... um ... print.  
- To save an image of a graph or chart, right-click on it and select Save Image as ...  

## Credits
This notebook activity was designed by [Quarknet](https://quarknet.org/) Neutrino Fellow Mike Plucinski.   

The data is the **HTRU2** data set, collected during the High Time Resolution Universe survey and donated to the [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/372/htru2) by Dr. Robert Lyon of the University of Manchester. It is used here under a [Creative Commons Attribution 4.0 International (CC BY 4.0)](https://creativecommons.org/licenses/by/4.0/) license.  

Please cite: Lyon, R. (2015). *HTRU2* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C5DK6R  

Thanks to the [scikit-learn](https://scikit-learn.org/) developers, and to the great folks at [Binder](https://mybinder.org/) and [Google Colaboratory](https://colab.research.google.com/notebooks/intro.ipynb) for making this notebook interactive without you needing to download it or install [Jupyter](https://jupyter.org/) on your own device. Find more activities and license info at [CODINGinK12.org](http://www.codingink12.org).